# Week 4 — Monday: Reshaping Data — Melting and Pivoting

**DATA 202 · Calvin University**

> The LORD God took the man and put him in the garden of Eden to work it and keep it. — Genesis 2:15

**Today's theme:** tending a garden = constantly reorganizing the same harvest (baskets → plot totals → season sum) — nothing added, nothing lost, just rearranged. Same idea as **melting** and **pivoting**.

**The one question everything today comes back to:**

> **What is one row about?**

**Today's outline:**

- Load and inspect the data
- Wide vs. Long: What Is One Row About?
- Part 1 — Melting: Wide to Long (SLO 04C)
- Part 2 — Pivoting and Exploding (SLO 04C)
- Careful with Reshaping + what's next

**Watch for:** 🎯 Predict First (guess before we run) · 🙋 Quick Check (verbal, no code)</cell id="9325b7c0-228e-431a-9af1-6ab6f5cd5bdb">

---
## Loading the Data · ~5 min</cell id="ecb0167e-f7ef-42bd-bc71-129db8530caf">

In [ ]:
import pandas as pd

DATA_PATH = "../../datasets/plot_yields.csv"
yields = pd.read_csv(DATA_PATH)
yields.head()

In [ ]:
yields.info()

**What to notice:**
- 18 rows — one per garden plot
- 6 columns (`Week1_lbs`...`Week6_lbs`) — one week's harvest weight each, side by side
- `Crops` crams multiple values into one cell (`"lettuce, squash"`) — same multi-value problem as Practice 02

🙋 **Quick Check:** Finish out loud: *"Right now, one row of `yields` is about ___."* Then: total harvest across all 18 plots, **Week 3 only** — can one `.groupby()` call get you that right now? Why not? (Hint: what would you even group *by*?)

---
## Wide vs. Long: What Is One Row About? · ~8 min

Before any code — a small example, not the real 18-plot table yet.

Picture 2 plots, 3 weeks. The same 6 harvest numbers, two different table shapes:

![A wide table with one row per plot and separate Week1/Week2/Week3 columns, next to a long table with one row per plot-per-week and a single Week column plus a single Harvest_lbs column. Arrows labeled "melt" and "pivot" connect the two, with the caption: one row = one plot's whole season (wide) vs. one row = one plot, in one week (long).](images/wide_long_diagram.png)

**Same 6 numbers in both tables** — nothing added, nothing lost. Only the answer to *"what is one row about?"* changed.

* **Wide:** one row = **one plot's whole season**. G01's Week 2 harvest → row `G01`, column `Week2_lbs`.
* **Long:** one row = **one plot, in one week**. Same number → row where `Plot_ID`=`G01` *and* `Week`=`Week2`, column `Harvest_lbs`.

Same fact, two different addresses.

🙋 **Quick Check:** Which shape for *"every week's harvest for plot G01, side by side"*? Which for *"average harvest per plot-week, across the garden"*? Not right vs. wrong — each built for a different question.

**`melt()`**: column *names* → values in a new column; their contents → values in another new column.
**`pivot_table()`**: reverse — values back into column names.

→ doing both next, on the real 18-plot data. Predict *what is one row about?* before each output.

---
## Part 1: Melting — Wide to Long (SLO 04C) · ~19 min

Right now: one row = **one plot's whole season**. Melt the six `Week*_lbs` columns into one `Week` column + one `Harvest_lbs` column.

🎯 **Predict First:** After melting `yields`, what will one row be about? Finish it: *"One row will be one ___, in one ___."* Then predict the row count: 18 plots × 6 weeks = ?

In [ ]:
long = pd.melt(
    yields,
    id_vars=["Plot_ID", "Gardener", "Crops"],
    value_vars=["Week1_lbs", "Week2_lbs", "Week3_lbs", "Week4_lbs", "Week5_lbs", "Week6_lbs"],
    var_name="Week",
    value_name="Harvest_lbs",
)
long["Week"] = long["Week"].str.replace("_lbs", "", regex=False)
long.shape

In [ ]:
long[long['Plot_ID'] == 'G01']

**Check your prediction:** one row is now **one plot, in one week** — 108 rows (melting multiplies row count by columns unstacked).

- `id_vars` → columns **repeated** on every row (still identify the plot)
- `value_vars` → columns **unstacked**: their *names* become `Week` values, their *contents* become `Harvest_lbs` values

🙋 **Quick Check:** total harvest across all 18 plots, every week — now that "week" lives inside the data, is this a `.groupby()` you already know how to write?

In [ ]:
long.groupby('Week', sort=False)['Harvest_lbs'].sum().round(1)

Season peaks Week 3 (194.8 lbs total), tapers by Week 6 (67.5 lbs) — **unaskable** while weeks were six separate columns. That's the whole point of melting: some questions only open up once "which week" lives *inside* the data.

---
### 🔨 Mini-Task A — Melt Just the Early Season (~3 min)

Predict first: melt only `Week1_lbs`, `Week2_lbs`, `Week3_lbs` — what's one row about? How many rows (18 plots × ? weeks)?

Then write it: melt those 3 columns into `early_long`, same `id_vars`, `var_name="Week"`, `value_name="Harvest_lbs"`. Check the shape.

In [ ]:
# Your code here
early_long = None


---
### 🔨 Task 1 — Find the Single Best Week (~5 min)

Using `long` (one row = one plot, one week): find the **single highest** `Harvest_lbs` across the whole season — which plot, which week, how many pounds.

1. `.idxmax()` on `long['Harvest_lbs']` → index of the largest value
2. `.loc[]` on that index → the full row
3. Assign `Plot_ID` → `best_plot`, `Week` → `best_week`, `Harvest_lbs` → `best_harvest`

*Works only because one row = one plot-in-one-week — `idxmax()` on the wide table's `Week3_lbs` column alone would only ever find the best Week 3. Same `idxmax()` + `.loc[]` pattern as Practice 01; what changed is what a row means.*

In [ ]:
# Your code here


---
## Part 2: Pivoting and Exploding (SLO 04C) · ~13 min

Melting: one row → one plot-week. **Pivoting** asks the opposite: what if we want one row = **one plot's whole season** again?

🎯 **Predict First:** Pivot `long` back — `Week` values become column headers again. What's one row about now? What shape do you expect — should it match `yields` exactly?

In [ ]:
back_to_wide = long.pivot_table(
    index=["Plot_ID", "Gardener", "Crops"],
    columns="Week",
    values="Harvest_lbs",
).reset_index()
back_to_wide.shape

`(18, 9)` — one row = "one plot's whole season" again, exact shape we started with.

- `index` → columns that identify a row
- `columns` → column whose *values* become new headers
- `values` → what fills the new cells

Melt and pivot are exact inverses — no information created or destroyed, only the row's meaning flips back and forth.

*Duplicate `Plot_ID` + `Week` rows would need squashing into one cell — default `aggfunc="mean"`. No duplicates here, but that's the safety net.*

---
### 🔨 Mini-Task B — Reverse a Partial Melt (~3 min)

Predict first: pivot `early_long` back to wide — one row means what again? Shape — 18 plots, plus how many columns?

Then write it: `.pivot_table()` (`index=["Plot_ID", "Gardener", "Crops"]`, `columns="Week"`, `values="Harvest_lbs"`) → `early_wide`. Check the shape.

In [ ]:
# Your code here


### One More Reshape: What If One Row Means One Crop?

- `Crops` crams multiple values into one cell (`"lettuce, squash"`) — same problem as `Pos` in Practice 02
- can't answer "how many plots grow lettuce?" without reading every cell by hand
- `.explode()` → gives each crop its own row:

In [ ]:
yields_crops = yields.copy()
yields_crops["Crop_List"] = yields_crops["Crops"].str.split(", ")
exploded = yields_crops.explode("Crop_List").reset_index(drop=True)
exploded.shape

In [ ]:
exploded['Crop_List'].value_counts()

One row of `exploded` = **one plot, growing one crop** → `.value_counts()` on `Crop_List` directly answers "how many plots grow this crop." Lettuce and kale tie for most widely-grown (7 of 18 plots each).

⚠️ **count of plots**, not pounds harvested — `.explode()` multiplied rows, not harvest weight.

---
### 🔨 Task 2 — Most Common Crop, Cross-Checked (~4 min)

1. `exploded['Crop_List'].value_counts()` — most common crop(s)? Tie for first? → `most_common_crop_count`
2. `exploded['Crop_List'].nunique()` — how many distinct crops? → `n_distinct_crops`

In [ ]:
# Your code here


---
## Careful with Reshaping · ~5 min

Every reshape today was the same move: **decide what a row should mean, then let `melt()`/`pivot_table()` get you there.** No information added or lost — but the row's meaning changes what a careless `.sum()`/`.mean()` silently computes:

* **wide**: summing `Week3_lbs` = Week 3's total across every plot — one cell per plot
* **long**: summing all of `Harvest_lbs` = the season total — but only correct because every plot contributed exactly 6 rows. Skipped weeks (vs. zero-harvest weeks) would quietly bias any "average per plot-week" toward whoever reported more often — the same imbalance from this week's reading.

**A table's shape is not neutral — neither is "what is one row about?"** Wide makes "compare weeks side by side" trivial, long flips that. Neither is *the* correct shape — only the one that fits the question.

**Still unanswerable either way:** is every one of these 18 plots registered with the garden coordinator? Needs a *second* table — a new question about what connects one table's rows to another's. → Wednesday.

---
## Coming Up

| Day | Topic | Builds on today |
|---|---|---|
| Wed | Joining tables — keys, primary/foreign keys, four join types | Same garden, a second table — "what is a row about," asked of *two* tables at once |
| Week 5 | Clustering & Dimensionality Reduction | Finding groups the data suggests, instead of ones we choose (`Plot_ID`, `Week`) in advance |</cell id="4edd2fc6-3847-44e6-8af7-5ddcceaf73c0">